# TRABAJO PRACTICO N°6

## 1- OBJETIVO

Implementar un algoritmo de busqueda para resolver la torre de Hanoi

ALGORITMO ELEGIDO VORAZ

## 2- IMPLEMENTACION VORAZ

### 2.1- ESTADO HANOI

Implementamos una clase llamada EstadoHanoi, la cual es la encargada de guardar el estado actual del hanoi, asi como el objetivo de la misma (que la cantidad total de discos esten en la varilla de destino)

In [19]:
import heapq


class EstadoHanoi:
    def __init__(self, varillas):
        
        self.varillas = varillas

    def __eq__(self, other):
        return isinstance(other, EstadoHanoi) and self.varillas == other.varillas

    def __hash__(self):
        return hash(self.varillas)

    def es_objetivo(self, num_discos, varilla_destino):
        return len(self.varillas[varilla_destino]) == num_discos

## 2.2- Heuristica


In [20]:
def heuristica_propia(estado, num_discos, varilla_destino):

    def calcular_h(disco, destino_actual):
        if disco == 0:
            return 0
        
        
        varilla_actual = -1
        for idx in range(3):
            if disco in estado.varillas[idx]:
                varilla_actual = idx
                break
        
        if varilla_actual == destino_actual:
            
            return calcular_h(disco - 1, destino_actual)
        else:
            varilla_aux = 3 - varilla_actual - destino_actual
            return (2 ** (disco - 1)) + calcular_h(disco - 1, varilla_aux)

    return calcular_h(num_discos, varilla_destino)

def obtener_acciones_validas(estado):
    acciones = []
    for i in range(3):
        if not estado.varillas[i]:
            continue
        disco_origen = estado.varillas[i][-1] 
        
        for j in range(3):
            if i == j:
                continue
            if not estado.varillas[j] or estado.varillas[j][-1] > disco_origen:
                acciones.append((i, j))
    return acciones


def aplicar_accion(estado, accion):
    
    origen, destino = accion
    nuevas = [list(v) for v in estado.varillas]
    disco = nuevas[origen].pop()
    nuevas[destino].append(disco)
    return EstadoHanoi(tuple(tuple(v) for v in nuevas))

## 2.3- Guardar los estados 

In [21]:
class Nodo:
    def __init__(self, estado, padre=None, accion=None, g=0, h=0):
        self.estado = estado
        self.padre = padre
        self.accion = accion
        self.g = g  
        self.h = h  

## 2.4- BUSQUEDA VORAZ

In [22]:

def busqueda_voraz(num_discos, varilla_origen=0, varilla_destino=1):
    
    varillas_ini = [[] for _ in range(3)]
    varillas_ini[varilla_origen] = list(range(num_discos, 0, -1))
    estado_inicial = EstadoHanoi(tuple(tuple(v) for v in varillas_ini))

    h_ini = heuristica_propia(estado_inicial, num_discos, varilla_destino)
    nodo_inicial = Nodo(estado_inicial, g=0, h=h_ini)

    frontera = []
    contador = 0
    heapq.heappush(frontera, (nodo_inicial.h, contador, nodo_inicial))
    
    visitados = {estado_inicial}
    nodos_expandidos = 0

    while frontera:
        
        _, _, nodo_actual = heapq.heappop(frontera)
        nodos_expandidos += 1

        
        if nodo_actual.estado.es_objetivo(num_discos, varilla_destino):
            camino = []
            curr = nodo_actual
            while curr:
                camino.append((curr.estado, curr.accion))
                curr = curr.padre
            camino.reverse()
            return camino, nodos_expandidos

        
        for accion in obtener_acciones_validas(nodo_actual.estado):
            nuevo_estado = aplicar_accion(nodo_actual.estado, accion)

            if nuevo_estado not in visitados:
                visitados.add(nuevo_estado)
                h = heuristica_propia(nuevo_estado, num_discos, varilla_destino)
                nuevo_nodo = Nodo(nuevo_estado, nodo_actual, accion, nodo_actual.g + 1, h)
                
                contador += 1
                heapq.heappush(frontera, (nuevo_nodo.h, contador, nuevo_nodo))

    return None, nodos_expandidos



if __name__ == "__main__":
    NUM_DISCOS = 5
    camino, expandidos = busqueda_voraz(NUM_DISCOS)
    
  
    print(" BÚSQUEDA VORAZ ")
    
    print(f"Cantidad de discos: {NUM_DISCOS}")
    
    
    print("Secuencia de pasos:")
    for paso, (estado, accion) in enumerate(camino):
        if accion is None:
            print(f"Paso {paso:2d} [ESTADO INICIAL] : Varillas -> {estado.varillas}")
        else:
            orig, dest = accion
            print(f" Mover {orig} -> {dest}     : Varillas -> {estado.varillas}")

 BÚSQUEDA VORAZ 
Cantidad de discos: 5
Secuencia de pasos:
Paso  0 [ESTADO INICIAL] : Varillas -> ((5, 4, 3, 2, 1), (), ())
 Mover 0 -> 1     : Varillas -> ((5, 4, 3, 2), (1,), ())
 Mover 0 -> 2     : Varillas -> ((5, 4, 3), (1,), (2,))
 Mover 1 -> 2     : Varillas -> ((5, 4, 3), (), (2, 1))
 Mover 0 -> 1     : Varillas -> ((5, 4), (3,), (2, 1))
 Mover 2 -> 0     : Varillas -> ((5, 4, 1), (3,), (2,))
 Mover 2 -> 1     : Varillas -> ((5, 4, 1), (3, 2), ())
 Mover 0 -> 1     : Varillas -> ((5, 4), (3, 2, 1), ())
 Mover 0 -> 2     : Varillas -> ((5,), (3, 2, 1), (4,))
 Mover 1 -> 2     : Varillas -> ((5,), (3, 2), (4, 1))
 Mover 1 -> 0     : Varillas -> ((5, 2), (3,), (4, 1))
 Mover 2 -> 0     : Varillas -> ((5, 2, 1), (3,), (4,))
 Mover 1 -> 2     : Varillas -> ((5, 2, 1), (), (4, 3))
 Mover 0 -> 1     : Varillas -> ((5, 2), (1,), (4, 3))
 Mover 0 -> 2     : Varillas -> ((5,), (1,), (4, 3, 2))
 Mover 1 -> 2     : Varillas -> ((5,), (), (4, 3, 2, 1))
 Mover 0 -> 1     : Varillas -> ((), (